In [6]:
# Basic initial check to count number of total patients and genes in the dataset.

import pandas as pd

BUCKET = "cancer-source-data"

df = pd.read_parquet(f"s3://{BUCKET}/processed/ml-ready/")

print(df['tumor_sample_barcode'].nunique(), "patients")
print(df['hugo_symbol'].nunique(), "genes")

473 patients
363 genes


In [7]:
# Checked to see if no patient has more than 1 cancer type assigned to them. Transforms the data into a patient x gene binary matrix.

labels = df[['tumor_sample_barcode', 'cancer_type']].drop_duplicates()
labels = labels.set_index('tumor_sample_barcode')['cancer_type']
assert df.groupby('tumor_sample_barcode')['cancer_type'].nunique().max() == 1

matrix = pd.crosstab(df['tumor_sample_barcode'], df['hugo_symbol'])
matrix = (matrix > 0).astype('int8')
matrix['cancer_type'] = labels

In [8]:
# Label encoded the cancer types

%pip install scikit-learn -q   # skip if already installed

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
matrix['label'] = le.fit_transform(matrix['cancer_type'])

label_map = dict(zip(le.transform(le.classes_), le.classes_))
print(label_map)

Note: you may need to restart the kernel to use updated packages.
{np.int64(0): 'BRCA', np.int64(1): 'COAD', np.int64(2): 'KIRC', np.int64(3): 'LUAD', np.int64(4): 'PRAD'}


In [9]:
# Checked counts for each label

print(matrix['cancer_type'].value_counts())

cancer_type
LUAD    98
COAD    97
BRCA    96
PRAD    91
KIRC    91
Name: count, dtype: int64


In [10]:
# Train test data split.

from sklearn.model_selection import train_test_split

gene_cols = [c for c in matrix.columns if c not in ('cancer_type', 'label')]

X = matrix[gene_cols]
y = matrix['label']
patient_ids = matrix.index

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, patient_ids,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(y_test.value_counts())

Train: 378, Test: 95
label
3    20
1    20
0    19
4    18
2    18
Name: count, dtype: int64


In [11]:
# Merged labels and features back into single dfs then saved them as CSV files to use later in the model training process.

train_df = pd.concat([y_train.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
test_df = pd.concat([y_test.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)

train_df.to_csv('train.csv', index=False, header=False)
test_df.to_csv('test.csv', index=False, header=False)

In [12]:
# Saved two reference files: the test patient IDs (to trace predictions back to real patients later).
# and the label mapping as JSON

import json

pd.Series(ids_test.values, name='tumor_sample_barcode').to_csv('test_patient_ids.csv', index=False)

with open('label_mapping.json', 'w') as f:
    json.dump({str(k): v for k, v in label_map.items()}, f, indent=2)

In [13]:
# uploaded all files to s3 bucket.

import boto3
s3 = boto3.client('s3')

for f in ['train.csv', 'test.csv', 'test_patient_ids.csv', 'label_mapping.json']:
    s3.upload_file(f, BUCKET, f'processed/model-input/{f}')

print(f"Uploaded to s3://{BUCKET}/processed/model-input/")

/Users/eesha/cancer-classifier/venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Uploaded to s3://cancer-source-data/processed/model-input/


In [14]:
# Basic final check to see if dimensions for csv file are correct 

check = pd.read_csv('train.csv', header=None)
print(check.shape)        # should be (≈378, 364) — label + 363 genes
print(check[0].value_counts())  # label distribution, should roughly match the full set

(378, 364)
0
3    78
1    77
0    77
2    73
4    73
Name: count, dtype: int64
